# Train E5-base — TripletLoss (hard negative v2) trên Colab

Fine-tune tiếp từ **`e5_base_finetuned_final`** (1 epoch) bằng **TripletLoss** trên:
`train_triplets_category_hardneg_v2.jsonl`

**Output:** `embedding_project/models/e5_base_finetuned_triplet_hardneg_final/`

### Quy trình (theo setup của bạn)
| Nguồn | Nội dung |
|---|---|
| **GitHub** | Code + triplets + script train |
| **Google Drive** | File zip model `e5_base_finetuned_final` → giải nén vào `embedding_project/models/` |

**Runtime:** chọn **GPU** (T4). ~180 triplets × 1 epoch ≈ vài phút.

### Trên Drive cần có
- `e5_base_finetuned_final.zip` (hoặc tên zip bạn đặt — sửa `DRIVE_MODEL_ZIP` ở cell 3)

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" "accelerate>=1.1.0" torch datasets pandas scikit-learn numpy tqdm

## 2) Clone code từ GitHub

In [ ]:
import os
import shutil
import subprocess
import sys

PROJECT_DIR = "/content/llm_provider_benchmarking"
REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"  # ← sửa nếu khác

if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
print("Cloned:", REPO_URL)

os.chdir(PROJECT_DIR)
scripts_dir = os.path.join(PROJECT_DIR, "embedding_project", "scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

print("cwd:", os.getcwd())
print("triplets:", os.path.isfile("embedding_project/data/train_triplets_category_hardneg_v2.jsonl"))

## 3) Mount Drive và giải nén base model (1 epoch)

Upload zip lên Drive trước, ví dụ:
`MyDrive/models/e5_base_finetuned_final.zip`

Zip có thể chứa:
- folder `e5_base_finetuned_final/` ở root, hoặc
- các file model trực tiếp (config.json, model.safetensors, …)

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# ← Sửa đường dẫn zip model trên Drive của bạn
DRIVE_MODEL_ZIP = "/content/drive/MyDrive/models/e5_base_finetuned_final.zip"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/models"  # nơi lưu model sau train

BASE_MODEL_DIR = Path(PROJECT_DIR) / "embedding_project" / "models" / "e5_base_finetuned_final"
BASE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

if not os.path.isfile(DRIVE_MODEL_ZIP):
    raise FileNotFoundError(
        f"Không thấy {DRIVE_MODEL_ZIP}\n"
        "Hãy upload e5_base_finetuned_final.zip lên Drive và sửa DRIVE_MODEL_ZIP."
    )

extract_tmp = Path("/content/_model_zip_extract")
if extract_tmp.exists():
    shutil.rmtree(extract_tmp)
extract_tmp.mkdir(parents=True)

with zipfile.ZipFile(DRIVE_MODEL_ZIP) as z:
    z.extractall(extract_tmp)

# Tìm folder chứa config.json (root model SentenceTransformer)
candidates = [p for p in extract_tmp.rglob("config.json") if "1_Pooling" not in str(p)]
if not candidates:
    raise FileNotFoundError("Zip không có config.json — kiểm tra lại file zip model")

src_dir = candidates[0].parent
if BASE_MODEL_DIR.exists():
    shutil.rmtree(BASE_MODEL_DIR)
shutil.copytree(src_dir, BASE_MODEL_DIR)

print("Extracted model ->", BASE_MODEL_DIR)
print("Files:", sorted(p.name for p in BASE_MODEL_DIR.iterdir())[:8], "...")

## 4) Kiểm tra GPU và file đầu vào

In [ ]:
import json
import torch
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_DIR) / "embedding_project"
BASE_MODEL_DIR = PROJECT_ROOT / "models" / "e5_base_finetuned_final"
TRIPLETS_PATH = PROJECT_ROOT / "data" / "train_triplets_category_hardneg_v2.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "models" / "e5_base_finetuned_triplet_hardneg_final"

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for label, p in [
    ("Base model 1ep", BASE_MODEL_DIR),
    ("Triplets v2", TRIPLETS_PATH),
]:
    ok = p.is_dir() if p.suffix == "" else p.is_file()
    print(f"{'OK' if ok else 'MISSING':7} {label}: {p}")
    if not ok:
        raise FileNotFoundError(str(p))

n_triplets = sum(1 for line in TRIPLETS_PATH.read_text(encoding="utf-8").splitlines() if line.strip())
print(f"Triplets: {n_triplets}")
print(f"Output -> {OUTPUT_DIR}")

## 5) Cấu hình train

Mặc định khớp script local: 1 epoch, batch 16, lr `1e-5`, margin `0.2`.

In [ ]:
EPOCHS = 1
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
LEARNING_RATE = 1e-5
TRIPLET_MARGIN = 0.2
WARMUP_RATIO = 0.1
MAX_SEQ_LENGTH = 512
USE_FP16 = torch.cuda.is_available()
MAX_TRIPLETS = None  # đặt số nguyên để smoke test, ví dụ 20

print(
    f"epochs={EPOCHS} batch={BATCH_SIZE} lr={LEARNING_RATE} "
    f"margin={TRIPLET_MARGIN} fp16={USE_FP16}"
)

## 6) Train TripletLoss

In [ ]:
import json
from math import ceil
from pathlib import Path

from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader

rows = []
with TRIPLETS_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

if MAX_TRIPLETS is not None:
    rows = rows[:MAX_TRIPLETS]

examples = [
    InputExample(texts=[r["anchor"], r["positive"], r["negative"]])
    for r in rows
    if r.get("anchor") and r.get("positive") and r.get("negative")
]
if not examples:
    raise ValueError("Không có triplet hợp lệ")

print(f"Training on {len(examples)} triplets from {BASE_MODEL_DIR}")

model = SentenceTransformer(str(BASE_MODEL_DIR))
model.max_seq_length = MAX_SEQ_LENGTH

train_dataloader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.TripletLoss(model=model, triplet_margin=TRIPLET_MARGIN)

total_steps = max(1, ceil(len(train_dataloader)) * EPOCHS)
warmup_steps = int(total_steps * WARMUP_RATIO)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    output_path=str(OUTPUT_DIR),
    show_progress_bar=True,
    use_amp=USE_FP16,
)

model.save(str(OUTPUT_DIR))
print("Saved:", OUTPUT_DIR)

## 7) Smoke test encode (tùy chọn)

In [ ]:
import numpy as np

m = SentenceTransformer(str(OUTPUT_DIR))
sample = rows[0]
texts = [
    f"query: {sample['anchor']}",
    f"passage: {sample['positive'][:200]}",
    f"passage: {sample['negative'][:200]}",
]
emb = m.encode(texts, convert_to_numpy=True)
emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)

sim_pos = float(emb[0] @ emb[1])
sim_neg = float(emb[0] @ emb[2])
print("anchor:", sample["anchor"][:80])
print(f"cos(query, positive)={sim_pos:.4f}  cos(query, negative)={sim_neg:.4f}")
print("OK" if sim_pos > sim_neg else "WARN: positive không gần hơn negative — kiểm tra lại data")

## 8) Lưu lên Drive và tải về máy

In [ ]:
import shutil
from datetime import datetime
from google.colab import files

stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_name = f"e5_base_triplet_hardneg_{stamp}.zip"
zip_base = f"/content/{zip_name.replace('.zip', '')}"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=OUTPUT_DIR)
print("Zip:", zip_path, f"({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

drive_out = Path(DRIVE_OUTPUT_DIR) / zip_name
drive_out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, drive_out)
print("Saved to Drive:", drive_out)

files.download(zip_path)

## 9) Bước tiếp theo trên máy local

1. Tải zip từ Colab hoặc lấy từ Drive: `e5_base_triplet_hardneg_*.zip`
2. Giải nén vào `embedding_project/models/e5_base_finetuned_triplet_hardneg_final/`
3. Index Qdrant hoặc chạy manual eval (`manual_eval_queries.csv`)
4. So sánh với `e5_base_finetuned_final` (1ep) và `e5_base_finetuned_2ep_final`